# DengAI-UNT: Protocolo Experimental F6-F10 con Optuna y GPU
## Comparación Rigurosa de 5 Arquitecturas Espacio-Temporales sobre Datos Reales de La Libertad

Este notebook ejecuta el protocolo experimental exigido por el **Anexo A de la Ficha Técnica SP-01**:
- **Modelos Evaluados**:
  1. Individual 1: XGBoost (Tabular)
  2. Individual 2: LSTM (Recurrente Temporal)
  3. Individual 3: ConvLSTM (Espacio-Temporal)
  4. Híbrido 1: CNN-LSTM (Convolucional + Recurrente)
  5. Híbrido 2: ConvLSTM-Attention (Mecanismo de atención espacial y temporal)
- **Optimización**:
  - 100 trials por modelo con Optuna (TPE Sampler + Hyperband/Median Pruner).
  - Presupuesto computacional comparable.
  - Optimización **exclusivamente sobre Train y Validación interna** (el conjunto de prueba Test jamás participa).
- **Estrategia de Validación**:
  - Walk-forward temporal (26 semanas de validación, 26 semanas de prueba final).
  - Escaladores ajustados únicamente con datos de entrenamiento.
- **Pruebas Estadísticas Obligatorias**:
  - Prueba de Friedman aplicada sobre **bloques independientes** (semanas epidemiológicas de prueba).
  - Comparaciones pareadas post-hoc de Wilcoxon con corrección de Holm.
  - Prueba de McNemar sobre discordancia de clasificación de riesgo.
  - Prueba de DeLong sobre curvas ROC pareadas.

In [ ]:
# 1. Verificación del entorno y aceleración por GPU
import torch
print(f"PyTorch version: {torch.__version__}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo disponible: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Clonación o sincronización del repositorio DengAI-UNT
!git clone https://github.com/tu-usuario/DengAI-UNT.git || (cd DengAI-UNT && git pull)
%cd DengAI-UNT
!pip install -q optuna shap xgboost scikit-learn pandas numpy matplotlib seaborn scipy

In [ ]:
# 3. Importación de módulos del pipeline DengAI
import sys
sys.path.insert(0, 'backend')

from app.db.session import SessionLocal
from app.ml.features import load_panel, build_dataset
from app.ml.models.base import ModelData
from app.ml.models.registry import create_model, SEARCH_SPACES
from app.ml.metrics import evaluate_all
from app.ml.stats_tests import compare_models, mcnemar_test, delong_roc_test

db = SessionLocal()
panel = load_panel(db, mode='auto')
print(f"Panel cargado: {len(panel.panel)} registros, {len(panel.districts)} distritos, {len(panel.weeks)} semanas")
print(f"Modo: {panel.data_mode} | Sintético: {panel.is_synthetic}")

# Construir dataset walk-forward sin fuga temporal
bundle = build_dataset(panel, horizon=1, test_weeks=26, validation_weeks=26)
print(f"Particiones: Train={len(bundle.y_train)}, Val={len(bundle.y_val)}, Test={len(bundle.y_test)}")

train_data = ModelData(bundle.X_train, bundle.seq_train, bundle.grid_train, bundle.cell_train, bundle.y_train, bundle.feature_names)
val_data = ModelData(bundle.X_val, bundle.seq_val, bundle.grid_val, bundle.cell_val, bundle.y_val, bundle.feature_names)
test_data = ModelData(bundle.X_test, bundle.seq_test, bundle.grid_test, bundle.cell_test, bundle.y_test, bundle.feature_names)

In [ ]:
# 4. Optimización con Optuna: 100 trials con TPE Sampler y Median Pruner por modelo
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

models = ['xgboost', 'lstm', 'convlstm', 'cnn_lstm', 'convlstm_attention']
studies = {}
best_params_all = {}
N_TRIALS = 100

for algo in models:
    print(f"\n>>> Optimizando hiperparámetros para {algo.upper()} ({N_TRIALS} trials)... <<<")
    space = SEARCH_SPACES[algo]
    
    def objective(trial):
        params = {}
        for name, spec in space.items():
            if spec['type'] == 'int':
                params[name] = trial.suggest_int(name, spec['low'], spec['high'], step=spec.get('step', 1))
            elif spec['type'] == 'float':
                params[name] = trial.suggest_float(name, spec['low'], spec['high'], log=spec.get('log', False))
            elif spec['type'] == 'categorical':
                params[name] = trial.suggest_categorical(name, spec['choices'])
        
        model = create_model(algo)
        model.fit(train_data, val_data, params=params, epochs=25)
        val_pred = model.predict(val_data)
        val_mae = float(np.mean(np.abs(bundle.y_val - val_pred)))
        return val_mae
    
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    studies[algo] = study
    best_params_all[algo] = study.best_params
    print(f"Mejor Val MAE ({algo}): {study.best_value:.4f}")

In [ ]:
# 5. Entrenamiento Final con Mejores Hiperparámetros y Evaluación en Test (Sin Fuga)
import numpy as np
import pandas as pd

test_predictions = {}
final_metrics = {}

for algo in models:
    print(f"Entrenando modelo final {algo.upper()}...")
    best_params = best_params_all[algo]
    model = create_model(algo)
    # Ajuste con train y validación
    model.fit(train_data, val_data, params=best_params, epochs=35)
    y_test_pred = np.clip(model.predict(test_data), 0, None)
    test_predictions[algo] = y_test_pred
    
    outbreak_th = float(np.quantile(bundle.y_train, 0.90))
    m = evaluate_all(bundle.y_test, y_test_pred, y_insample=bundle.y_train[-100:], outbreak_threshold=outbreak_th)
    final_metrics[algo] = m

df_metrics = pd.DataFrame(final_metrics).T
display(df_metrics[['MAE', 'RMSE', 'MASE', 'AUC_PR', 'sensitivity', 'specificity']])

In [ ]:
# 6. Pruebas Estadísticas Rigurosas (Anexo A: Friedman por Bloques + Wilcoxon-Holm + McNemar + DeLong)
test_blocks = bundle.meta_test['target_week'].astype(str).tolist()
stats_result = compare_models(test_predictions, bundle.y_test, blocks=test_blocks)

print("=== PRUEBA DE FRIEDMAN SOBRE BLOQUES INDEPENDIENTES ===")
print(f"Bloques evaluados: {stats_result['blocks']} semanas")
print(f"Estadístico Chi2: {stats_result['friedman']['statistic']:.4f}")
print(f"P-valor: {stats_result['friedman']['p_value']:.4e}")
print(f"Diferencia estadísticamente significativa: {stats_result['friedman']['significant']}")

print("\n=== COMPARACIONES PAREADAS DE WILCOXON CON CORRECCIÓN DE HOLM ===")
df_pairwise = pd.DataFrame(stats_result['pairwise'])
display(df_pairwise[['model_a', 'model_b', 'p_value_adjusted', 'significant', 'superior']])

In [ ]:
# 7. Generación de Gráficos de Publicación (IMRyD)
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot de errores absolutos por modelo across test blocks
errors_dict = {algo: np.abs(bundle.y_test - test_predictions[algo]) for algo in models}
df_err = pd.DataFrame(errors_dict)
sns.boxplot(data=df_err, ax=axes[0], palette='Set2')
axes[0].set_title('Distribución de Errores Absolutos en Conjunto de Prueba', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Error Absoluto (Casos)')

# Curvas de Convergencia de Optuna
for algo, st in studies.items():
    vals = [t.value for t in st.trials if t.value is not None]
    best_so_far = np.minimum.accumulate(vals)
    axes[1].plot(best_so_far, label=algo.upper())
axes[1].set_title('Convergencia de Optimización Bayesiana (Optuna TPE)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Trial')
axes[1].set_ylabel('Mejor MAE de Validación')
axes[1].legend()

plt.tight_layout()
plt.savefig('comparacion_modelos_optuna.png', dpi=300)
plt.show()